In [1]:
import pandas as pd

df= pd.read_csv("ecommerce_dataset.csv", names=["category", "description"], header=None)
print(df.shape)
df.head(3)

(50425, 2)


,category,description
0,Household,Paper Plane Design Framed Wall Hanging Motivat...
1,Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
2,Household,SAF 'UV Textured Modern Art Print Framed' Pain...


In [2]:
df.dropna(inplace=True)
df.shape

(50424, 2)

In [3]:
df.category.unique()

array(['Household', 'Books', 'Clothing & Accessories', 'Electronics'],
      dtype=object)

In [4]:
df.category.replace("Clothing & Accessories", "Clothing_Accessories", inplace=True)

/var/folders/7h/hhfb2zp15xn9_lf52_bb379h0000gn/T/ipykernel_1675/2912538940.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.category.replace("Clothing & Accessories", "Clothing_Accessories", inplace=True)


In [5]:
df.category.unique()

array(['Household', 'Books', 'Clothing_Accessories', 'Electronics'],
      dtype=object)

In [6]:
df['category'] = '__label__' + df['category'].astype(str)
df.head(5)

,category,description
0,__label__Household,Paper Plane Design Framed Wall Hanging Motivat...
1,__label__Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
2,__label__Household,SAF 'UV Textured Modern Art Print Framed' Pain...
3,__label__Household,"SAF Flower Print Framed Painting (Synthetic, 1..."
4,__label__Household,Incredible Gifts India Wooden Happy Birthday U...


In [7]:
df['category_description'] = df['category'] + ' ' + df['description']
df.head(3)

,category,description,category_description
0,__label__Household,Paper Plane Design Framed Wall Hanging Motivat...,__label__Household Paper Plane Design Framed W...
1,__label__Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ...",__label__Household SAF 'Floral' Framed Paintin...
2,__label__Household,SAF 'UV Textured Modern Art Print Framed' Pain...,__label__Household SAF 'UV Textured Modern Art...


In [8]:
import re

text = "  VIKI's | Bookcase/Bookshelf (3-Shelf/Shelve, White) | ? . hi"
text = re.sub(r'[^\w\s\']',' ', text)
text = re.sub(' +', ' ', text)
text.strip().lower()

"viki's bookcase bookshelf 3 shelf shelve white hi"

In [9]:
def preprocess(text):
    text = re.sub(r'[^\w\s\']',' ', text)
    text = re.sub(' +', ' ', text)
    return text.strip().lower() 

In [10]:
df['category_description'] = df['category_description'].map(preprocess)
df.head()

,category,description,category_description
0,__label__Household,Paper Plane Design Framed Wall Hanging Motivat...,__label__household paper plane design framed w...
1,__label__Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ...",__label__household saf 'floral' framed paintin...
2,__label__Household,SAF 'UV Textured Modern Art Print Framed' Pain...,__label__household saf 'uv textured modern art...
3,__label__Household,"SAF Flower Print Framed Painting (Synthetic, 1...",__label__household saf flower print framed pai...
4,__label__Household,Incredible Gifts India Wooden Happy Birthday U...,__label__household incredible gifts india wood...


In [11]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2)

In [12]:

train.shape, test.shape

((40339, 3), (10085, 3))

In [13]:
train.head(3)

,category,description,category_description
43760,__label__Electronics,JmGO 3D DLP Link 3D Active Shutter Glasses for...,__label__electronics jmgo 3d dlp link 3d activ...
25115,__label__Books,Objective Science and Technology,__label__books objective science and technology
30829,__label__Books,The Power of Counseling: A Travel with My Life...,__label__books the power of counseling a trave...


In [14]:
train.to_csv("ecommerce.train", columns=["category_description"], index=False, header=False)
test.to_csv("ecommerce.test", columns=["category_description"], index=False, header=False)

In [14]:
import fasttext

model = fasttext.train_supervised(input="ecommerce.train")
model.test("ecommerce.test")

Read 4M words
Number of words:  79499
Number of labels: 4
Progress: 100.0% words/sec/thread: 2081245 lr:  0.000000 avg.loss:  0.194993 ETA:   0h 0m 0s


(10083, 0.9707428344738669, 0.9707428344738669)

In [15]:
model.predict("wintech assemble desktop pc cpu 500 gb sata hdd 4 gb ram intel c2d processor 3")

(('__label__electronics',), array([0.99862003]))

In [16]:
model.predict("ockey men's cotton t shirt fabric details 80 cotton 20 polyester super combed cotton rich fabric")

(('__label__clothing_accessories',), array([1.00001001]))

In [17]:
model.predict("think and grow rich deluxe edition")

(('__label__books',), array([1.00000978]))

In [18]:

model.get_nearest_neighbors("painting")

[(0.9985684156417847, 'bxke1801in'),
 (0.9985474944114685, 'conglobate'),
 (0.9985474944114685, 'descriptionseasoning'),
 (0.9985474944114685, 'dredges'),
 (0.9985349774360657, 'homecrafts'),
 (0.9985269904136658, 'microwable'),
 (0.9985269904136658, 'devnow'),
 (0.9985083937644958, '74inch'),
 (0.9985083937644958, '95x'),
 (0.998497724533081, 'ft12')]

In [22]:
model.get_nearest_neighbors("banglore")

[(0.0, 'of'),
 (0.0, 'a'),
 (0.0, 'for'),
 (0.0, 'in'),
 (0.0, 'is'),
 (0.0, '</s>'),
 (0.0, 'albus'),
 (0.0, 'girlsrecommended'),
 (0.0, 'kindergartners'),
 (0.0, 'cardboarddimensions')]

In [23]:
model.get_nearest_neighbors("apple")

[(0.9958951473236084, '30x52'),
 (0.9929959774017334, 'iphone'),
 (0.9876549243927002, 'her'),
 (0.9873791933059692, 'lightweight'),
 (0.9862409830093384, 'preparedness'),
 (0.9862409830093384, '50lm'),
 (0.9859263896942139, '320g'),
 (0.9859263896942139, '447g'),
 (0.9859263896942139, 'dioptric'),
 (0.9840101599693298, 'qhm636')]

In [24]:
model.get_nearest_neighbors("samsung")

[(0.9937505125999451, 'power'),
 (0.9921004176139832, 'press'),
 (0.9894958138465881, 'speed'),
 (0.9871193766593933, 'center'),
 (0.9868972301483154, 'having'),
 (0.9848331809043884, 'surface'),
 (0.9837538003921509, 'release'),
 (0.9837037324905396, 'green'),
 (0.9821711182594299, 'plug'),
 (0.9821237325668335, 'makes')]

In [26]:
model.predict("think and grow rich")

(('__label__books',), array([1.00000906]))